# RSNA Knee — Kaggle runner

Thin wrapper. All logic lives in the repo's `src/`, which arrives here as an
attached Kaggle Dataset imported from GitHub.

**Order matters:** refresh the dataset on Kaggle *before* re-running, or this
executes the previous revision. Cell 2 prints the revision so a stale run is
obvious instead of silent.

Set `SCRIPT` in the last cell to pick the job:

| `SCRIPT`  | what it does                                   | Internet | Accelerator |
|-----------|------------------------------------------------|----------|-------------|
| `extract` | report → 12 labels, one time over 4407 reports  | **on**   | GPU T4 x2   |
| `eval`    | grade those labels against the 58 gold studies  | off      | CPU is fine |
| `train`   | image model → `submission.csv`                  | off      | GPU T4 x2   |

`extract` must run *before* `eval`, and `eval` reads the CSV from
`/kaggle/working`, which does not survive the session — so run them together,
or save the output (see the last cell).

Use T4, not P100 — Kaggle's PyTorch ships without Pascal kernels.


In [ ]:
# Find the code. Normally it arrives as an attached Kaggle Dataset, so search
# /kaggle/input first. Kaggle nests inputs by type
# (/kaggle/input/competitions/..., /kaggle/input/datasets/<owner>/...), so this
# walks a few levels instead of guessing a fixed path. The walk is depth-capped
# and prunes the DICOM trees on purpose: an unbounded walk would crawl tens of
# thousands of series directories before giving up.
#
# If nothing is attached, clone from GitHub instead. That works while Internet
# is on -- which the extract job needs anyway -- and means a code change reaches
# Kaggle by a plain `git push`. It will NOT work for the offline submission run,
# which still needs the dataset attached.
import os
import subprocess

REPO_URL = "https://github.com/xuanxiaoyuan/RSNA-Knee-Abnormality-Detection.git"
MARKER = "src/train.py"
HEAVY = {"train_series", "test_series", "train", "test"}


def find_repo(root, max_depth=4):
    base_depth = root.rstrip("/").count("/")
    for dirpath, dirnames, _ in os.walk(root):
        if dirpath.rstrip("/").count("/") - base_depth >= max_depth:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if d not in HEAVY]
        if os.path.exists(os.path.join(dirpath, MARKER)):
            return dirpath
    return None


print("attached under /kaggle/input:")
for entry in sorted(os.listdir("/kaggle/input")):
    path = f"/kaggle/input/{entry}"
    if os.path.isdir(path):
        print(f"  {entry}/  ({len(os.listdir(path))} entries)")

CODE = find_repo("/kaggle/input")

if CODE is None:
    CODE = "/kaggle/working/rsne-code"
    if os.path.exists(f"{CODE}/.git"):
        print(f"\n[code] refreshing clone at {CODE}")
        subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
    else:
        print(f"\n[code] nothing attached; cloning {REPO_URL}")
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
        except subprocess.CalledProcessError:
            raise SystemExit(
                "[code] git clone failed. Either enable Internet "
                "(Settings -> Internet) or attach the code dataset via Add Input."
            )

print("\nCODE =", CODE)


In [ ]:
# Staleness check: does this match the VERSION you just pushed?
print(open(f"{CODE}/VERSION").read())

In [ ]:
# Pick the job, then run it. Each entry point reads its own config from the repo.
SCRIPT = "extract"  # "extract", "train", or "eval"

import runpy, sys

# Re-running a cell does NOT reload imported modules: Python caches them in
# sys.modules, and cell 1's `git pull` only changes files on disk. So a second
# run in the same kernel mixes revisions — train.py is re-executed by runpy and
# picks up the new code, while dataset/model/utils stay at whatever was first
# imported, which shows up as an unpacking error deep inside the run. Drop them
# so the run always matches the code cell 1 just pulled.
for name in ("dataset", "model", "utils", "extract_labels", "eval_extractor"):
    stale = sys.modules.pop(name, None)
    if stale is not None:
        print(f"[cache] dropped stale {name} module: {getattr(stale, '__file__', '?')}")

if SCRIPT == "extract":
    # One-time report -> label job. Needs Internet ON for the model weights.
    # --limit 20 is a smoke test: prove the weights download and the 12-bit
    # answer parses, before spending hours on all 4407. Raise the limit in steps.
    argv = ["extract_labels.py",
            "--config", f"{CODE}/configs/extract.yaml",
            "--limit", "20"]
elif SCRIPT == "eval":
    # Scores labels_derived.csv against the 58 gold studies. Reads the CSV from
    # the work dir by default, so it only works in the session that made it.
    argv = ["eval_extractor.py", "--config", f"{CODE}/configs/extract.yaml"]
elif SCRIPT == "train":
    argv = ["train.py", "--config", f"{CODE}/configs/base.yaml"]
else:
    raise SystemExit(f"unknown SCRIPT {SCRIPT!r} - use 'extract', 'eval' or 'train'")

print("running", *argv)
sys.argv = argv
runpy.run_path(f"{CODE}/src/{argv[0]}", run_name="__main__")


## `extract` — the one-time label job

1. Internet **on**, accelerator T4 x2.
2. Run the smoke test (`--limit 20`) and read the printed
   `[parse] exact=… loose=… fail=…` line and the s/report timing before
   committing to the full run. A `fail` count above zero is worth inspecting.
3. Raise `--limit` in steps (20 → 300 → all). `/kaggle/working` does **not**
   survive a session, so after each run use **Save Version → Save & Run All**,
   or copy `labels_derived.csv` into a Kaggle Dataset. `resume: true` skips
   studies already in the file, so a killed session continues rather than
   restarts — but only if the file survives, which is the whole point of saving.
4. Delete `labels_derived.csv` before the final full run if the smoke test
   used a different model or config, so stale rows cannot hide in the output.

## `train` — after the debug run is clean

1. Set `debug: false` in `configs/base.yaml` (or pass `--no-debug`).
2. Bump `VERSION`, push, refresh the Kaggle dataset.
3. Confirm the printed marker changed, then run full.

Full runs need checkpointing — a session is capped at ~12h and gets killed
at the limit, so never leave the only copy of a trained model in memory.
